# Framingham 10-year CHD risk factors

**Original Question:** What are the strongest risk factors for developing coronary heart disease within ten years in the framingham dataset?

_Exported from PlotStudio_

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# PlotStudio stub — no-op outside the app
def add_to_workspace(df):
    pass


# Load source data
df_framingham = pd.read_csv("data/df_framingham.csv")

## Task 1: Establish a clean, well-documented modeling cohort and understand crude associations between baseline factors and 10-year CHD.

**Acceptance Criteria:** A modeling-ready dataframe with clear sample definition, event counts, and basic descriptive and unadjusted association results for all major risk factors, enabling informed specification of multivariable models.

### 1.1: Create a cleaned df_framingham_model suitable for logistic regression and risk-factor analysis.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Work from a copy to preserve the original dataframe
df_framingham_model = df_framingham.copy()

# Modeling-ready type conversions
for col in ["education", "cigsPerDay", "BPMeds"]:
    if col in df_framingham_model.columns:
        df_framingham_model[col] = pd.to_numeric(
            df_framingham_model[col], errors="coerce"
        )

# Create glucose missingness indicator before imputation
if "glucose" in df_framingham_model.columns:
    df_framingham_model["glucose_was_missing"] = (
        df_framingham_model["glucose"].isna().astype(int)
    )
    df_framingham_model["glucose"] = pd.to_numeric(
        df_framingham_model["glucose"], errors="coerce"
    )
    df_framingham_model["glucose"] = df_framingham_model["glucose"].fillna(
        df_framingham_model["glucose"].median()
    )

# Document row/event counts before low-missingness row dropping
n_rows_before = len(df_framingham_model)
events_before = (
    int(df_framingham_model["TenYearCHD"].sum())
    if "TenYearCHD" in df_framingham_model.columns
    else np.nan
)

# Drop rows only for low-missingness columns that are critical for modeling
low_missing_cols = [
    col
    for col in [
        "male",
        "age",
        "currentSmoker",
        "BPMeds",
        "prevalentStroke",
        "prevalentHyp",
        "diabetes",
        "totChol",
        "sysBP",
        "diaBP",
        "BMI",
        "heartRate",
        "TenYearCHD",
    ]
    if col in df_framingham_model.columns
]
df_framingham_model = df_framingham_model.dropna(subset=low_missing_cols).copy()

# Re-apply safe type conversions after dropping rows
for col in [
    "male",
    "currentSmoker",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "TenYearCHD",
    "glucose_was_missing",
]:
    if col in df_framingham_model.columns:
        df_framingham_model[col] = pd.to_numeric(
            df_framingham_model[col], errors="coerce"
        ).astype(int)

for col in [
    "education",
    "BPMeds",
    "cigsPerDay",
    "age",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
]:
    if col in df_framingham_model.columns:
        df_framingham_model[col] = pd.to_numeric(
            df_framingham_model[col], errors="coerce"
        ).astype(float)

# Document row/event counts after dropping low-missingness rows
n_rows_after = len(df_framingham_model)
events_after = (
    int(df_framingham_model["TenYearCHD"].sum())
    if "TenYearCHD" in df_framingham_model.columns
    else np.nan
)

# Outlier counts retained using clinically plausible ranges from the observed dataset description
outlier_specs = {
    "age": (32, 70),
    "cigsPerDay": (0, 70),
    "totChol": (100, 600),
    "sysBP": (80, 300),
    "diaBP": (40, 200),
    "BMI": (15, 50),
    "heartRate": (40, 150),
    "glucose": (40, 400),
}
outlier_counts = {}
for col, (lo, hi) in outlier_specs.items():
    if col in df_framingham_model.columns:
        outlier_counts[col] = int(
            ((df_framingham_model[col] < lo) | (df_framingham_model[col] > hi)).sum()
        )

# Concise cleaning summary
print("Framingham modeling dataset cleaning summary")
print(f"Rows before drop: {n_rows_before}")
print(f"CHD events before drop: {events_before}")
print(f"Rows after drop: {n_rows_after}")
print(f"CHD events after drop: {events_after}")
print("Outliers retained within clinically plausible ranges:")
for col, count in outlier_counts.items():
    print(f"{col}: {count}")

# Save to workspace
add_to_workspace(df_framingham_model)


### 1.2: Summarize baseline characteristics overall and by 10-year CHD status.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use existing modeling-ready dataframe directly

df_profile = df_framingham_model.copy()

total_n = len(df_profile)
total_events = int(df_profile["TenYearCHD"].sum())
total_incidence = total_events / total_n if total_n else np.nan

print("Cohort profile for 10-year CHD analysis")
print(f"Total sample size: {total_n:,}")
print(f"10-year CHD events: {total_events:,} ({total_incidence:.1%})")

continuous_vars = [
    "age",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
]
binary_vars = [
    "male",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "glucose_was_missing",
]
categorical_vars = ["education"]

print("")
print("Continuous variables: overall mean (SD), and by CHD status mean (SD)")
for col in continuous_vars:
    overall_mean = df_profile[col].mean()
    overall_sd = df_profile[col].std()
    mean_0 = df_profile.loc[df_profile["TenYearCHD"] == 0, col].mean()
    sd_0 = df_profile.loc[df_profile["TenYearCHD"] == 0, col].std()
    mean_1 = df_profile.loc[df_profile["TenYearCHD"] == 1, col].mean()
    sd_1 = df_profile.loc[df_profile["TenYearCHD"] == 1, col].std()
    diff = mean_1 - mean_0
    print(
        f"{col}: overall {overall_mean:.2f} ({overall_sd:.2f}); no-CHD {mean_0:.2f} ({sd_0:.2f}); CHD {mean_1:.2f} ({sd_1:.2f}); diff {diff:+.2f}"
    )

print("")
print("Binary variables: count and proportion overall, and by CHD status")
for col in binary_vars:
    overall_count = int(df_profile[col].sum())
    overall_prop = df_profile[col].mean()
    count_0 = int(df_profile.loc[df_profile["TenYearCHD"] == 0, col].sum())
    prop_0 = df_profile.loc[df_profile["TenYearCHD"] == 0, col].mean()
    count_1 = int(df_profile.loc[df_profile["TenYearCHD"] == 1, col].sum())
    prop_1 = df_profile.loc[df_profile["TenYearCHD"] == 1, col].mean()
    diff = prop_1 - prop_0
    print(
        f"{col}: overall {overall_count:,} ({overall_prop:.1%}); no-CHD {count_0:,} ({prop_0:.1%}); CHD {count_1:,} ({prop_1:.1%}); diff {diff:+.1%}"
    )

print("")
print("Categorical variable: education distribution overall and by CHD status")
for col in categorical_vars:
    df_tmp = df_profile.groupby(["TenYearCHD", col]).size().reset_index(name="count")
    df_tmp["group_total"] = df_tmp.groupby("TenYearCHD")["count"].transform("sum")
    df_tmp["proportion"] = df_tmp["count"] / df_tmp["group_total"]
    df_pivot = df_tmp.pivot(
        index=col, columns="TenYearCHD", values=["count", "proportion"]
    ).fillna(0)
    df_pivot.columns = [f"{a}_{b}" for a, b in df_pivot.columns]
    df_pivot = df_pivot.reset_index()
    print(f"{col}:")
    print(df_pivot.to_string(index=False))

# Identify clearest crude differences between CHD and non-CHD groups
crude_continuous = []
for col in continuous_vars:
    mean_0 = df_profile.loc[df_profile["TenYearCHD"] == 0, col].mean()
    mean_1 = df_profile.loc[df_profile["TenYearCHD"] == 1, col].mean()
    crude_continuous.append((col, abs(mean_1 - mean_0), mean_1 - mean_0))

crude_binary = []
for col in binary_vars:
    prop_0 = df_profile.loc[df_profile["TenYearCHD"] == 0, col].mean()
    prop_1 = df_profile.loc[df_profile["TenYearCHD"] == 1, col].mean()
    crude_binary.append((col, abs(prop_1 - prop_0), prop_1 - prop_0))

crude_continuous = sorted(crude_continuous, key=lambda x: x[1], reverse=True)
crude_binary = sorted(crude_binary, key=lambda x: x[1], reverse=True)

print("")
print("Variables with the clearest crude differences between CHD and non-CHD groups")
print("Top continuous differences:")
for col, _, signed_diff in crude_continuous[:5]:
    direction = "higher" if signed_diff > 0 else "lower"
    print(f"{col}: CHD group is {direction} by {abs(signed_diff):.2f} units")
print("Top binary differences:")
for col, _, signed_diff in crude_binary[:5]:
    direction = "higher" if signed_diff > 0 else "lower"
    print(f"{col}: CHD group is {direction} by {abs(signed_diff):.1%}")


### 1.3: Quantify unadjusted associations between each key risk factor and 10-year CHD.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

# Work from the existing modeling-ready dataframe
df_model = df_framingham_model.copy()

# Ensure modeling-safe types
for col in [
    "male",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "TenYearCHD",
    "glucose_was_missing",
]:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce").astype(int)

for col in [
    "education",
    "age",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
]:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# Helper for one-predictor logistic regression


def fit_logit_unadjusted(df_in, predictor, outcome="TenYearCHD", scale=1.0, label=None):
    df_use = df_in[[outcome, predictor]].dropna().copy()
    n = len(df_use)
    events = int(df_use[outcome].sum()) if n else 0
    non_events = n - events
    abs_risk = events / n if n else np.nan
    if n == 0 or df_use[predictor].nunique(dropna=True) < 2:
        return {
            "factor": label if label is not None else predictor,
            "contrast": label if label is not None else predictor,
            "n": n,
            "events": events,
            "absolute_risk": abs_risk,
            "or": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "p_value": np.nan,
            "model_status": "insufficient variation",
        }
    X = sm.add_constant(df_use[[predictor]].astype(float), has_constant="add")
    y = df_use[outcome].astype(float)
    try:
        model = sm.Logit(y, X).fit(disp=0)
        coef = model.params[predictor] * scale
        se = model.bse[predictor] * scale
        or_val = float(np.exp(coef))
        ci_lower = float(np.exp(coef - 1.96 * se))
        ci_upper = float(np.exp(coef + 1.96 * se))
        p_value = float(model.pvalues[predictor])
        status = "ok"
    except Exception:
        or_val = np.nan
        ci_lower = np.nan
        ci_upper = np.nan
        p_value = np.nan
        status = "fit failed"
    return {
        "factor": label if label is not None else predictor,
        "contrast": label if label is not None else predictor,
        "n": n,
        "events": events,
        "absolute_risk": abs_risk,
        "or": or_val,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "p_value": p_value,
        "model_status": status,
    }


results_rows = []

# Binary risk factors
for col in [
    "male",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "glucose_was_missing",
]:
    if col in df_model.columns:
        res1 = fit_logit_unadjusted(df_model, col, label=col)
        res1["type"] = "binary"
        res1["direction"] = "1 vs 0"
        results_rows.append(res1)

# Continuous risk factors scaled per 10 units
for col in [
    "age",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
]:
    if col in df_model.columns:
        res1 = fit_logit_unadjusted(
            df_model, col, scale=10.0, label=f"{col} per 10 units"
        )
        res1["type"] = "continuous"
        res1["direction"] = "per 10-unit increase"
        results_rows.append(res1)

# Clinically grouped age categories
if "age" in df_model.columns:
    df_age_cat = df_model.copy()
    df_age_cat["age_group"] = pd.cut(
        df_age_cat["age"],
        bins=[0, 45, 55, 65, np.inf],
        labels=["<45", "45-54", "55-64", "65+"],
        right=False,
        include_lowest=True,
    )
    age_dummies = pd.get_dummies(df_age_cat["age_group"], drop_first=True)
    if "<45" in df_age_cat["age_group"].astype(str).unique():
        baseline_age = "<45"
    else:
        baseline_age = "lowest category"
    for col in age_dummies.columns:
        df_tmp = (
            pd.concat([df_age_cat[["TenYearCHD"]].dropna(), age_dummies[[col]]], axis=1)
            .dropna()
            .copy()
        )
        df_tmp[col] = df_tmp[col].astype(float)
        n = len(df_tmp)
        events = int(df_tmp["TenYearCHD"].sum())
        abs_risk = events / n if n else np.nan
        X = sm.add_constant(df_tmp[[col]], has_constant="add")
        try:
            model = sm.Logit(df_tmp["TenYearCHD"].astype(float), X).fit(disp=0)
            or_val = float(np.exp(model.params[col]))
            ci = model.conf_int().loc[col]
            ci_lower = float(np.exp(ci[0]))
            ci_upper = float(np.exp(ci[1]))
            p_value = float(model.pvalues[col])
            status = "ok"
        except Exception:
            or_val = np.nan
            ci_lower = np.nan
            ci_upper = np.nan
            p_value = np.nan
            status = "fit failed"
        results_rows.append(
            {
                "factor": "age category",
                "contrast": f"{col} vs {baseline_age}",
                "type": "categorical",
                "direction": "vs reference",
                "n": n,
                "events": events,
                "absolute_risk": abs_risk,
                "or": or_val,
                "ci_lower": ci_lower,
                "ci_upper": ci_upper,
                "p_value": p_value,
                "model_status": status,
            }
        )

# Clinically grouped BMI categories
if "BMI" in df_model.columns:
    df_bmi_cat = df_model.copy()
    df_bmi_cat["bmi_group"] = pd.cut(
        df_bmi_cat["BMI"],
        bins=[0, 18.5, 25, 30, np.inf],
        labels=["<18.5", "18.5-24.9", "25.0-29.9", "30+"],
        right=False,
        include_lowest=True,
    )
    bmi_dummies = pd.get_dummies(df_bmi_cat["bmi_group"], drop_first=True)
    baseline_bmi = "<18.5"
    for col in bmi_dummies.columns:
        df_tmp = (
            pd.concat([df_bmi_cat[["TenYearCHD"]].dropna(), bmi_dummies[[col]]], axis=1)
            .dropna()
            .copy()
        )
        df_tmp[col] = df_tmp[col].astype(float)
        n = len(df_tmp)
        events = int(df_tmp["TenYearCHD"].sum())
        abs_risk = events / n if n else np.nan
        X = sm.add_constant(df_tmp[[col]], has_constant="add")
        try:
            model = sm.Logit(df_tmp["TenYearCHD"].astype(float), X).fit(disp=0)
            or_val = float(np.exp(model.params[col]))
            ci = model.conf_int().loc[col]
            ci_lower = float(np.exp(ci[0]))
            ci_upper = float(np.exp(ci[1]))
            p_value = float(model.pvalues[col])
            status = "ok"
        except Exception:
            or_val = np.nan
            ci_lower = np.nan
            ci_upper = np.nan
            p_value = np.nan
            status = "fit failed"
        results_rows.append(
            {
                "factor": "BMI category",
                "contrast": f"{col} vs {baseline_bmi}",
                "type": "categorical",
                "direction": "vs reference",
                "n": n,
                "events": events,
                "absolute_risk": abs_risk,
                "or": or_val,
                "ci_lower": ci_lower,
                "ci_upper": ci_upper,
                "p_value": p_value,
                "model_status": status,
            }
        )

# Combine and rank results

df_results = pd.DataFrame(results_rows)
df_results["effect_size"] = np.where(
    df_results["or"].notna(), np.maximum(df_results["or"], 1 / df_results["or"]), np.nan
)
df_results = df_results.sort_values(
    by=["effect_size", "p_value", "n"], ascending=[False, True, False]
).reset_index(drop=True)

# Format for display

df_results_display = df_results.copy()
df_results_display["absolute_risk"] = (df_results_display["absolute_risk"] * 100).round(
    1
).astype(str) + "%"
df_results_display["or_ci"] = df_results_display.apply(
    lambda r: (
        f"{r['or']:.2f} ({r['ci_lower']:.2f}, {r['ci_upper']:.2f})"
        if pd.notna(r["or"])
        else "NA"
    ),
    axis=1,
)
df_results_display["p_value"] = df_results_display["p_value"].apply(
    lambda x: f"{x:.3g}" if pd.notna(x) else "NA"
)
df_results_display["effect_size"] = df_results_display["effect_size"].round(2)

# Select and order columns for a concise combined table

df_results_display = df_results_display[
    [
        "factor",
        "contrast",
        "type",
        "n",
        "events",
        "absolute_risk",
        "or_ci",
        "p_value",
        "effect_size",
        "model_status",
    ]
]

print("Unadjusted logistic regression results for 10-year CHD risk factors")
print(df_results_display.to_string(index=False))
print("")
print(
    "Stability note: sparse factors such as prevalentStroke and BPMeds have few events, so their odds ratios may be unstable and confidence intervals wide."
)
print(
    "Age and BMI categories are shown with clinically meaningful cut points; continuous predictors are scaled to per 10-unit increases."
)


### 1.4: Visually compare 10-year CHD incidence across key clinically meaningful categories.
_Output: figure_

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Build ordered systolic BP categories and combine with age group labels

df_bp_chart = df_framingham_model.copy()

df_bp_chart["age_group"] = pd.cut(
    df_bp_chart["age"],
    bins=[0, 45, 55, 65, np.inf],
    labels=["<45", "45-54", "55-64", "65+"],
    right=False,
    include_lowest=True,
)

df_bp_chart["sbp_group"] = pd.cut(
    df_bp_chart["sysBP"],
    bins=[0, 120, 140, 160, np.inf],
    labels=["<120", "120-139", "140-159", "160+"],
    right=False,
    include_lowest=True,
)

# Ensure explicit ordering for plotting
sbp_order = ["<120", "120-139", "140-159", "160+"]
age_order = ["<45", "45-54", "55-64", "65+"]

df_bp_chart["age_group"] = pd.Categorical(
    df_bp_chart["age_group"], categories=age_order, ordered=True
)
df_bp_chart["sbp_group"] = pd.Categorical(
    df_bp_chart["sbp_group"], categories=sbp_order, ordered=True
)
df_bp_chart["smoking_status"] = np.where(
    df_bp_chart["currentSmoker"] == 1, "Current smoker", "Non-smoker"
)

# Aggregate CHD incidence within clinically meaningful strata

df_bp_summary = (
    df_bp_chart.groupby(["age_group", "sbp_group", "smoking_status"], observed=True)
    .agg(n=("TenYearCHD", "size"), events=("TenYearCHD", "sum"))
    .reset_index()
)
df_bp_summary["chd_incidence"] = df_bp_summary["events"] / df_bp_summary["n"]
df_bp_summary["category_label"] = (
    df_bp_summary["age_group"].astype(str)
    + " | SBP "
    + df_bp_summary["sbp_group"].astype(str)
)

# Sort labels so the clinical progression is clear
category_order = []
for age_label in age_order:
    for sbp_label in sbp_order:
        category_order.append(f"{age_label} | SBP {sbp_label}")

df_bp_summary["category_label"] = pd.Categorical(
    df_bp_summary["category_label"], categories=category_order, ordered=True
)
df_bp_summary = df_bp_summary.sort_values(
    ["age_group", "sbp_group", "smoking_status"]
).reset_index(drop=True)

color_map = {"Non-smoker": "#4C78A8", "Current smoker": "#E45756"}

fig = px.bar(
    df_bp_summary,
    x="category_label",
    y="chd_incidence",
    color="smoking_status",
    barmode="group",
    category_orders={
        "category_label": category_order,
        "smoking_status": ["Non-smoker", "Current smoker"],
    },
    color_discrete_map=color_map,
    labels={
        "category_label": "Age group | Systolic BP category",
        "chd_incidence": "10-year CHD incidence",
        "smoking_status": "Smoking status",
    },
    title="10-year CHD risk rises across higher systolic BP categories, especially among older adults and smokers",
)
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.update_yaxes(tickformat=",.0%")
fig.update_layout(
    xaxis_title="Age group | Systolic BP category", yaxis_title="10-year CHD incidence"
)
fig.show()


## Task 2: Identify the strongest independent baseline risk factors for 10-year CHD, explore nonlinear thresholds, and quantify the impact of risk factor combinations and subgroup differences.

**Acceptance Criteria:** A set of multivariable logistic regression results with adjusted odds ratios and CIs, evidence on nonlinear/threshold behavior for key continuous risk factors, quantified effects of risk-factor clustering and key interactions (e.g., hypertension+diabetes, sex and age), and at least one visualization of predicted risk patterns.

_Mode: exploratory_

### 2.1: Fit and refine a core multivariable logistic regression model to estimate independent effects of major risk factors on 10-year CHD.
_Output: exploration_

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Start from the cleaned modeling cohort already in the namespace
df_model_core = df_framingham_model.copy()

# Ensure modeling-safe types
for col in [
    "male",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "TenYearCHD",
    "glucose_was_missing",
]:
    if col in df_model_core.columns:
        df_model_core[col] = pd.to_numeric(df_model_core[col], errors="coerce").astype(
            int
        )

for col in [
    "education",
    "age",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
]:
    if col in df_model_core.columns:
        df_model_core[col] = pd.to_numeric(df_model_core[col], errors="coerce")

# Helper: fit logistic regression and return model plus analysis frame


def fit_logit_model(df_in, predictors, outcome="TenYearCHD"):
    df_use = df_in[[outcome] + predictors].dropna().copy()
    X = sm.add_constant(df_use[predictors].astype(float), has_constant="add")
    y = df_use[outcome].astype(float)
    model = sm.Logit(y, X).fit(disp=0)
    return model, df_use, X


# Full clinically interpretable specification
full_predictors = [
    "age",
    "male",
    "education",
    "currentSmoker",
    "cigsPerDay",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "glucose",
    "glucose_was_missing",
]

# Compute VIFs for the fuller specification

df_vif_input = df_model_core[full_predictors].dropna().copy()
df_vif_matrix = sm.add_constant(df_vif_input.astype(float), has_constant="add")
df_vif_rows = []
for i, col in enumerate(df_vif_matrix.columns):
    if col == "const":
        continue
    df_vif_rows.append(
        {
            "variable": col,
            "vif": float(variance_inflation_factor(df_vif_matrix.values, i)),
        }
    )
df_vif = (
    pd.DataFrame(df_vif_rows).sort_values("vif", ascending=False).reset_index(drop=True)
)

print("Multicollinearity check for the fuller specification")
print(df_vif.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Preferred simplifications based on obvious redundancy and interpretability
# - Keep systolic BP and drop diastolic BP because they are strongly correlated and SBP is typically more predictive for CHD
# - Keep glucose and glucose_was_missing (missingness indicator for the original missing values)
# - Keep cigsPerDay and drop currentSmoker because amount smoked is more informative than binary smoking status
# - Keep prevalence history and medication variables because they are clinically distinct
preferred_predictors = [
    "age",
    "male",
    "education",
    "cigsPerDay",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "totChol",
    "sysBP",
    "BMI",
    "glucose",
    "glucose_was_missing",
]

# Compare fuller vs simpler specification
model_full, df_use_full, X_full = fit_logit_model(df_model_core, full_predictors)
model_preferred, df_use_pref, X_pref = fit_logit_model(
    df_model_core, preferred_predictors
)

print("Model comparison")
print(
    f"Full model: n={int(model_full.nobs)}, AIC={model_full.aic:.2f}, BIC={model_full.bic:.2f}, LL={model_full.llf:.2f}"
)
print(
    f"Preferred model: n={int(model_preferred.nobs)}, AIC={model_preferred.aic:.2f}, BIC={model_preferred.bic:.2f}, LL={model_preferred.llf:.2f}"
)

# Use the preferred model as the final clinically interpretable specification
model_final = model_preferred

# Build adjusted odds ratios table
params = model_final.params
conf = model_final.conf_int()
df_core_model_or = pd.DataFrame(
    {
        "comparison": [
            "Age (per 10 years)",
            "Male (vs female)",
            "Education (per 1 level increase)",
            "Cigarettes per day (per 10 cigarettes)",
            "BP medication use (yes vs no)",
            "Prior stroke history (yes vs no)",
            "Prevalent hypertension (yes vs no)",
            "Diabetes (yes vs no)",
            "Total cholesterol (per 10 mg/dL)",
            "Systolic BP (per 10 mmHg)",
            "BMI (per 10 kg/m^2)",
            "Glucose (per 10 mg/dL)",
            "Glucose missing at baseline (yes vs no)",
        ],
        "term": [
            "age",
            "male",
            "education",
            "cigsPerDay",
            "BPMeds",
            "prevalentStroke",
            "prevalentHyp",
            "diabetes",
            "totChol",
            "sysBP",
            "BMI",
            "glucose",
            "glucose_was_missing",
        ],
    }
)

scale_map = {
    "age": 10.0,
    "male": 1.0,
    "education": 1.0,
    "cigsPerDay": 10.0,
    "BPMeds": 1.0,
    "prevalentStroke": 1.0,
    "prevalentHyp": 1.0,
    "diabetes": 1.0,
    "totChol": 10.0,
    "sysBP": 10.0,
    "BMI": 10.0,
    "glucose": 10.0,
    "glucose_was_missing": 1.0,
}

or_values = []
ci_lowers = []
ci_uppers = []
p_values = []
for term in df_core_model_or["term"]:
    scale = scale_map[term]
    beta = params[term] * scale
    se = model_final.bse[term] * scale
    or_values.append(float(np.exp(beta)))
    ci_lowers.append(float(np.exp(beta - 1.96 * se)))
    ci_uppers.append(float(np.exp(beta + 1.96 * se)))
    p_values.append(float(model_final.pvalues[term]))

df_core_model_or["adjusted_or"] = or_values
df_core_model_or["ci_lower"] = ci_lowers
df_core_model_or["ci_upper"] = ci_uppers
df_core_model_or["p_value"] = p_values
df_core_model_or["n"] = int(model_final.nobs)
df_core_model_or["events"] = int(df_use_pref["TenYearCHD"].sum())
df_core_model_or["model"] = "preferred multivariable logistic regression"

df_core_model_or = df_core_model_or[
    [
        "comparison",
        "term",
        "adjusted_or",
        "ci_lower",
        "ci_upper",
        "p_value",
        "n",
        "events",
        "model",
    ]
].copy()
df_core_model_or = df_core_model_or.sort_values(
    "adjusted_or", ascending=False
).reset_index(drop=True)

# Human-readable formatting for reporting

df_core_model_or["or_ci"] = df_core_model_or.apply(
    lambda r: f"{r['adjusted_or']:.2f} ({r['ci_lower']:.2f}, {r['ci_upper']:.2f})",
    axis=1,
)
df_core_model_or["p_value_display"] = df_core_model_or["p_value"].apply(
    lambda x: f"{x:.3g}"
)

df_core_model_or_report = df_core_model_or[
    ["comparison", "or_ci", "p_value_display"]
].copy()

aic_gap = model_full.aic - model_final.aic
bic_gap = model_full.bic - model_final.bic
print("Preferred adjusted model summary")
print(f"Observations used: {int(model_final.nobs)}")
print(f'Events used: {int(df_use_pref["TenYearCHD"].sum())}')
print(f"AIC improvement vs full model: {aic_gap:.2f}")
print(f"BIC improvement vs full model: {bic_gap:.2f}")
print("Adjusted odds ratios from the preferred model")
print(df_core_model_or_report.to_string(index=False))

# Save the final reporting table to workspace
add_to_workspace(df_core_model_or)


### 2.2: Determine whether major continuous risk factors show nonlinear or threshold effects on 10-year CHD risk, especially at clinical cut points.
_Output: exploration_

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2
from sklearn.metrics import roc_auc_score

# Work from the existing modeling-ready cohort without modifying it in place
df_model_nonlinear = df_framingham_model.copy()

# Ensure safe numeric types
for col in [
    "TenYearCHD",
    "male",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "glucose_was_missing",
]:
    if col in df_model_nonlinear.columns:
        df_model_nonlinear[col] = pd.to_numeric(
            df_model_nonlinear[col], errors="coerce"
        ).astype(int)
for col in [
    "age",
    "education",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
]:
    if col in df_model_nonlinear.columns:
        df_model_nonlinear[col] = pd.to_numeric(
            df_model_nonlinear[col], errors="coerce"
        )

# Clinically meaningful categories
if "age" in df_model_nonlinear.columns:
    df_model_nonlinear["age_group"] = pd.cut(
        df_model_nonlinear["age"],
        bins=[0, 45, 55, 65, np.inf],
        labels=["<45", "45-54", "55-64", "65+"],
        right=False,
        include_lowest=True,
    )
if "sysBP" in df_model_nonlinear.columns:
    df_model_nonlinear["sbp_group"] = pd.cut(
        df_model_nonlinear["sysBP"],
        bins=[0, 120, 140, 160, np.inf],
        labels=["<120", "120-139", "140-159", "160+"],
        right=False,
        include_lowest=True,
    )
if "totChol" in df_model_nonlinear.columns:
    df_model_nonlinear["chol_group"] = pd.cut(
        df_model_nonlinear["totChol"],
        bins=[0, 200, 240, np.inf],
        labels=["<200", "200-239", "240+"],
        right=False,
        include_lowest=True,
    )
if "BMI" in df_model_nonlinear.columns:
    df_model_nonlinear["bmi_group"] = pd.cut(
        df_model_nonlinear["BMI"],
        bins=[0, 18.5, 25, 30, np.inf],
        labels=["<18.5", "18.5-24.9", "25.0-29.9", "30+"],
        right=False,
        include_lowest=True,
    )
if "glucose" in df_model_nonlinear.columns:
    df_model_nonlinear["glucose_group"] = pd.cut(
        df_model_nonlinear["glucose"],
        bins=[0, 100, 126, np.inf],
        labels=["<100", "100-125", "126+"],
        right=False,
        include_lowest=True,
    )

# Crude incidence tables for each categorical predictor
cat_specs = [
    ("age_group", "Age group"),
    ("sbp_group", "Systolic BP group"),
    ("chol_group", "Total cholesterol group"),
    ("bmi_group", "BMI group"),
    ("glucose_group", "Glucose group"),
]

df_incidence_rows = []
for col, label in cat_specs:
    if col in df_model_nonlinear.columns:
        df_tmp = df_model_nonlinear[[col, "TenYearCHD"]].dropna().copy()
        df_summary = (
            df_tmp.groupby(col, observed=True)
            .agg(n=("TenYearCHD", "size"), events=("TenYearCHD", "sum"))
            .reset_index()
        )
        df_summary["incidence"] = df_summary["events"] / df_summary["n"]
        df_summary["factor"] = label
        df_incidence_rows.append(df_summary)

if df_incidence_rows:
    df_incidence = pd.concat(df_incidence_rows, ignore_index=True).reset_index(
        drop=True
    )
else:
    df_incidence = pd.DataFrame(columns=["factor", "n", "events", "incidence"])

# Make category ordering explicit for clean output
order_map = {
    "age_group": ["<45", "45-54", "55-64", "65+"],
    "sbp_group": ["<120", "120-139", "140-159", "160+"],
    "chol_group": ["<200", "200-239", "240+"],
    "bmi_group": ["<18.5", "18.5-24.9", "25.0-29.9", "30+"],
    "glucose_group": ["<100", "100-125", "126+"],
}

print("Crude 10-year CHD incidence across clinically meaningful categories")
for col, label in cat_specs:
    if col in df_model_nonlinear.columns:
        print(label + ":")
        df_show = (
            df_model_nonlinear[[col, "TenYearCHD"]]
            .dropna()
            .groupby(col, observed=True)
            .agg(n=("TenYearCHD", "size"), events=("TenYearCHD", "sum"))
            .reset_index()
        )
        df_show["incidence"] = df_show["events"] / df_show["n"]
        if col in order_map:
            df_show[col] = pd.Categorical(
                df_show[col], categories=order_map[col], ordered=True
            )
            df_show = df_show.sort_values(col)
        df_show["incidence"] = (df_show["incidence"] * 100).round(1).astype(str) + "%"
        print(df_show.to_string(index=False))
        print("")

# Helper for logistic regression models


def fit_logit(df_in, predictors, outcome="TenYearCHD"):
    df_use = df_in[[outcome] + predictors].dropna().copy()
    X = sm.add_constant(df_use[predictors].astype(float), has_constant="add")
    y = df_use[outcome].astype(float)
    model = sm.Logit(y, X).fit(disp=0)
    return model, df_use


# Linear preferred specification from the prior task
linear_predictors = [
    "age",
    "male",
    "education",
    "cigsPerDay",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "totChol",
    "sysBP",
    "BMI",
    "glucose",
    "glucose_was_missing",
]
model_linear, df_use_linear = fit_logit(df_model_nonlinear, linear_predictors)

# Categorical alternative replacing key linear terms with clinically meaningful groups
categorical_base_predictors = [
    "male",
    "education",
    "cigsPerDay",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "glucose_was_missing",
]

# Age categorical model
if "age_group" in df_model_nonlinear.columns:
    df_age_model = pd.get_dummies(
        df_model_nonlinear[
            ["TenYearCHD"] + categorical_base_predictors + ["age_group"]
        ].dropna(),
        columns=["age_group"],
        drop_first=True,
    )
    model_age_cat, df_use_age_cat = fit_logit(
        df_age_model, [c for c in df_age_model.columns if c != "TenYearCHD"]
    )
else:
    model_age_cat, df_use_age_cat = None, pd.DataFrame()

# SBP categorical model
if "sbp_group" in df_model_nonlinear.columns:
    df_sbp_model = pd.get_dummies(
        df_model_nonlinear[
            ["TenYearCHD"] + categorical_base_predictors + ["age", "sbp_group"]
        ].dropna(),
        columns=["sbp_group"],
        drop_first=True,
    )
    model_sbp_cat, df_use_sbp_cat = fit_logit(
        df_sbp_model, [c for c in df_sbp_model.columns if c != "TenYearCHD"]
    )
else:
    model_sbp_cat, df_use_sbp_cat = None, pd.DataFrame()

# Combined nonlinear model replacing age and systolic BP with categories
if (
    "age_group" in df_model_nonlinear.columns
    and "sbp_group" in df_model_nonlinear.columns
):
    df_combo_cat = pd.get_dummies(
        df_model_nonlinear[
            ["TenYearCHD"] + categorical_base_predictors + ["age_group", "sbp_group"]
        ].dropna(),
        columns=["age_group", "sbp_group"],
        drop_first=True,
    )
    model_combo_cat, df_use_combo_cat = fit_logit(
        df_combo_cat, [c for c in df_combo_cat.columns if c != "TenYearCHD"]
    )
else:
    model_combo_cat, df_use_combo_cat = None, pd.DataFrame()

# Nonlinear age check with quadratic term
if "age" in df_model_nonlinear.columns:
    df_age_quad = df_model_nonlinear[["TenYearCHD"] + linear_predictors].dropna().copy()
    df_age_quad["age_sq"] = df_age_quad["age"] ** 2
    age_quad_predictors = [
        "age",
        "age_sq",
        "male",
        "education",
        "cigsPerDay",
        "BPMeds",
        "prevalentStroke",
        "prevalentHyp",
        "diabetes",
        "totChol",
        "sysBP",
        "BMI",
        "glucose",
        "glucose_was_missing",
    ]
    model_age_quad, df_use_age_quad = fit_logit(df_age_quad, age_quad_predictors)
else:
    model_age_quad, df_use_age_quad = None, pd.DataFrame()

# Nonlinear SBP check with quadratic term
if "sysBP" in df_model_nonlinear.columns:
    df_sbp_quad = df_model_nonlinear[["TenYearCHD"] + linear_predictors].dropna().copy()
    df_sbp_quad["sysBP_sq"] = df_sbp_quad["sysBP"] ** 2
    sbp_quad_predictors = [
        "age",
        "male",
        "education",
        "cigsPerDay",
        "BPMeds",
        "prevalentStroke",
        "prevalentHyp",
        "diabetes",
        "totChol",
        "sysBP",
        "sysBP_sq",
        "BMI",
        "glucose",
        "glucose_was_missing",
    ]
    model_sbp_quad, df_use_sbp_quad = fit_logit(df_sbp_quad, sbp_quad_predictors)
else:
    model_sbp_quad, df_use_sbp_quad = None, pd.DataFrame()

# Compare fit statistics against linear model
comparison_rows = []
comparison_rows.append(
    {
        "model": "Linear preferred",
        "n": int(model_linear.nobs),
        "aic": model_linear.aic,
        "bic": model_linear.bic,
        "llf": model_linear.llf,
    }
)
if model_age_cat is not None:
    comparison_rows.append(
        {
            "model": "Age categorical",
            "n": int(model_age_cat.nobs),
            "aic": model_age_cat.aic,
            "bic": model_age_cat.bic,
            "llf": model_age_cat.llf,
        }
    )
if model_sbp_cat is not None:
    comparison_rows.append(
        {
            "model": "SBP categorical",
            "n": int(model_sbp_cat.nobs),
            "aic": model_sbp_cat.aic,
            "bic": model_sbp_cat.bic,
            "llf": model_sbp_cat.llf,
        }
    )
if model_combo_cat is not None:
    comparison_rows.append(
        {
            "model": "Age + SBP categorical",
            "n": int(model_combo_cat.nobs),
            "aic": model_combo_cat.aic,
            "bic": model_combo_cat.bic,
            "llf": model_combo_cat.llf,
        }
    )
if model_age_quad is not None:
    comparison_rows.append(
        {
            "model": "Age quadratic",
            "n": int(model_age_quad.nobs),
            "aic": model_age_quad.aic,
            "bic": model_age_quad.bic,
            "llf": model_age_quad.llf,
        }
    )
if model_sbp_quad is not None:
    comparison_rows.append(
        {
            "model": "SBP quadratic",
            "n": int(model_sbp_quad.nobs),
            "aic": model_sbp_quad.aic,
            "bic": model_sbp_quad.bic,
            "llf": model_sbp_quad.llf,
        }
    )

df_fit_compare = pd.DataFrame(comparison_rows).sort_values("aic").reset_index(drop=True)
df_fit_compare["delta_aic_vs_linear"] = df_fit_compare["aic"] - float(model_linear.aic)
df_fit_compare["delta_bic_vs_linear"] = df_fit_compare["bic"] - float(model_linear.bic)


# Likelihood ratio tests for nested comparisons when applicable
def lr_test(full_model, reduced_model):
    lr = 2 * (full_model.llf - reduced_model.llf)
    df_diff = int(full_model.df_model - reduced_model.df_model)
    p = chi2.sf(lr, df_diff)
    return lr, df_diff, p


print("Model fit comparison against the linear preferred specification")
print(df_fit_compare.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print("")

# Nested test summaries
if model_age_cat is not None:
    lr_stat, df_diff, pval = lr_test(model_age_cat, model_linear)
    print(f"Age categorical vs linear: LR={lr_stat:.2f}, df={df_diff}, p={pval:.4g}")
if model_sbp_cat is not None:
    lr_stat, df_diff, pval = lr_test(model_sbp_cat, model_linear)
    print(f"SBP categorical vs linear: LR={lr_stat:.2f}, df={df_diff}, p={pval:.4g}")
if model_combo_cat is not None:
    lr_stat, df_diff, pval = lr_test(model_combo_cat, model_linear)
    print(
        f"Age+SBP categorical vs linear: LR={lr_stat:.2f}, df={df_diff}, p={pval:.4g}"
    )
if model_age_quad is not None:
    lr_stat, df_diff, pval = lr_test(model_age_quad, model_linear)
    print(f"Age quadratic vs linear: LR={lr_stat:.2f}, df={df_diff}, p={pval:.4g}")
if model_sbp_quad is not None:
    lr_stat, df_diff, pval = lr_test(model_sbp_quad, model_linear)
    print(f"SBP quadratic vs linear: LR={lr_stat:.2f}, df={df_diff}, p={pval:.4g}")

# Concise interpretation of linearity vs threshold behavior
linearity_notes = []
if model_age_cat is not None:
    lr_stat, df_diff, pval = lr_test(model_age_cat, model_linear)
    if pval < 0.05:
        linearity_notes.append(
            "Age shows some nonlinearity/threshold behavior; the categorical age model fits better than the linear age term."
        )
    else:
        linearity_notes.append(
            "Age behaves roughly linearly; categorical age does not materially improve fit over a linear term."
        )
if model_sbp_cat is not None:
    lr_stat, df_diff, pval = lr_test(model_sbp_cat, model_linear)
    if pval < 0.05:
        linearity_notes.append(
            "Systolic BP shows threshold-like risk increases at higher categories."
        )
    else:
        linearity_notes.append(
            "Systolic BP behaves roughly linearly across the observed range."
        )
if model_combo_cat is not None:
    lr_stat, df_diff, pval = lr_test(model_combo_cat, model_linear)
    if pval < 0.05:
        linearity_notes.append(
            "Replacing age and SBP with categories improves fit, suggesting at least one of these predictors is not perfectly linear."
        )
    else:
        linearity_notes.append(
            "Replacing age and SBP with categories does not clearly improve fit, supporting approximate linearity."
        )

# Identify threshold-like jumps in the crude incidence tables
threshold_notes = []
for col, label in [
    ("sbp_group", "SBP"),
    ("glucose_group", "glucose"),
    ("chol_group", "cholesterol"),
    ("bmi_group", "BMI"),
    ("age_group", "age"),
]:
    if col in df_model_nonlinear.columns:
        df_tmp = df_model_nonlinear[[col, "TenYearCHD"]].dropna().copy()
        df_tmp = (
            df_tmp.groupby(col, observed=True)
            .agg(n=("TenYearCHD", "size"), events=("TenYearCHD", "sum"))
            .reset_index()
        )
        df_tmp["incidence"] = df_tmp["events"] / df_tmp["n"]
        if col in order_map:
            df_tmp[col] = pd.Categorical(
                df_tmp[col], categories=order_map[col], ordered=True
            )
            df_tmp = df_tmp.sort_values(col)
        inc_vals = df_tmp["incidence"].to_numpy()
        if len(inc_vals) >= 2 and np.nanmax(np.diff(inc_vals)) > 0.03:
            threshold_notes.append(
                f"{label} shows a noticeable step-up in crude incidence at higher categories."
            )

print("")
print("Summary interpretation")
for note in linearity_notes:
    print(note)
for note in threshold_notes[:5]:
    print(note)
if not threshold_notes:
    print(
        "No strong threshold jumps were evident in the crude categorical incidence tables."
    )


### 2.3: Quantify how combinations and clustering of risk factors (e.g., hypertension plus diabetes, multiple comorbidities) amplify 10-year CHD risk.
_Output: exploration_

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Start from the existing modeling-ready cohort
_df_cluster = df_framingham_model.copy()

# Safe type conversion
for _col in [
    "TenYearCHD",
    "male",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "glucose_was_missing",
]:
    if _col in _df_cluster.columns:
        _df_cluster[_col] = pd.to_numeric(_df_cluster[_col], errors="coerce").astype(
            int
        )
for _col in [
    "age",
    "education",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
]:
    if _col in _df_cluster.columns:
        _df_cluster[_col] = pd.to_numeric(_df_cluster[_col], errors="coerce")

# Build cluster flags: smoking, hypertension, diabetes, obesity
_df_cluster["smoking_flag"] = (_df_cluster["currentSmoker"] == 1).astype(int)
_df_cluster["hypertension_flag"] = (
    (_df_cluster["prevalentHyp"] == 1) | (_df_cluster["BPMeds"] == 1)
).astype(int)
_df_cluster["diabetes_flag"] = (_df_cluster["diabetes"] == 1).astype(int)
_df_cluster["obesity_flag"] = (_df_cluster["BMI"] >= 30).astype(int)

_df_cluster["risk_count"] = _df_cluster[
    ["smoking_flag", "hypertension_flag", "diabetes_flag", "obesity_flag"]
].sum(axis=1)

# Crude incidence and unadjusted ORs by collapsed risk-count levels
_df_riskcount = (
    _df_cluster.groupby("risk_count", observed=True)
    .agg(n=("TenYearCHD", "size"), events=("TenYearCHD", "sum"))
    .reset_index()
    .sort_values("risk_count")
    .reset_index(drop=True)
)
_df_riskcount["incidence"] = _df_riskcount["events"] / _df_riskcount["n"]
_df_riskcount["odds"] = _df_riskcount["events"] / (
    _df_riskcount["n"] - _df_riskcount["events"]
)

_df_riskcount_ref = (
    _df_riskcount.loc[_df_riskcount["risk_count"] == 0, "odds"].iloc[0]
    if (_df_riskcount["risk_count"] == 0).any()
    else np.nan
)
_df_riskcount["unadjusted_or_vs_0"] = _df_riskcount["odds"] / _df_riskcount_ref

print("Clustered-risk analysis: smoking, hypertension, diabetes, obesity")
print("Crude CHD incidence and unadjusted ORs by number of risk factors")
print(
    _df_riskcount[
        ["risk_count", "n", "events", "incidence", "unadjusted_or_vs_0"]
    ].to_string(
        index=False,
        formatters={
            "incidence": lambda x: f"{x:.3f}",
            "unadjusted_or_vs_0": lambda x: f"{x:.2f}" if pd.notna(x) else "NA",
        },
    )
)

# Age- and sex-adjusted logistic model for risk-count levels
_df_riskcount_model = (
    _df_cluster[["TenYearCHD", "age", "male", "risk_count"]].dropna().copy()
)
_df_riskcount_model["risk_count_cat"] = pd.Categorical(
    _df_riskcount_model["risk_count"], categories=[0, 1, 2, 3, 4], ordered=True
)
_df_riskcount_dum = pd.get_dummies(
    _df_riskcount_model, columns=["risk_count_cat"], drop_first=True
)
_df_riskcount_X = sm.add_constant(
    _df_riskcount_dum.drop(columns=["TenYearCHD", "risk_count"]), has_constant="add"
).astype(float)
_df_riskcount_y = _df_riskcount_dum["TenYearCHD"].astype(float)
_df_riskcount_adj_model = sm.Logit(_df_riskcount_y, _df_riskcount_X).fit(disp=0)

_df_riskcount_terms = [
    c for c in _df_riskcount_X.columns if c.startswith("risk_count_cat_")
]
_df_riskcount_adj_rows = []
for _term in _df_riskcount_terms:
    _level = _term.replace("risk_count_cat_", "")
    _beta = float(_df_riskcount_adj_model.params[_term])
    _se = float(_df_riskcount_adj_model.bse[_term])
    _df_riskcount_adj_rows.append(
        {
            "level_vs_0": f"{_level} vs 0",
            "adjusted_or": float(np.exp(_beta)),
            "ci_lower": float(np.exp(_beta - 1.96 * _se)),
            "ci_upper": float(np.exp(_beta + 1.96 * _se)),
            "p_value": float(_df_riskcount_adj_model.pvalues[_term]),
        }
    )
_df_riskcount_adj = pd.DataFrame(_df_riskcount_adj_rows)

print("")
print("Age- and sex-adjusted ORs for risk-count levels")
if len(_df_riskcount_adj) > 0:
    print(
        _df_riskcount_adj.to_string(
            index=False,
            formatters={
                "adjusted_or": lambda x: f"{x:.2f}",
                "ci_lower": lambda x: f"{x:.2f}",
                "ci_upper": lambda x: f"{x:.2f}",
                "p_value": lambda x: f"{x:.3g}",
            },
        )
    )
else:
    print("No higher risk-count categories beyond the reference level were available.")

# Four-level hypertension/diabetes combination factor
_df_joint = _df_cluster.copy()
_df_joint["htn_dm_combo"] = np.select(
    [
        (_df_joint["hypertension_flag"] == 0) & (_df_joint["diabetes_flag"] == 0),
        (_df_joint["hypertension_flag"] == 1) & (_df_joint["diabetes_flag"] == 0),
        (_df_joint["hypertension_flag"] == 0) & (_df_joint["diabetes_flag"] == 1),
        (_df_joint["hypertension_flag"] == 1) & (_df_joint["diabetes_flag"] == 1),
    ],
    ["Neither", "Hypertension only", "Diabetes only", "Both"],
    default="Neither",
)
_df_joint["htn_dm_combo"] = pd.Categorical(
    _df_joint["htn_dm_combo"],
    categories=["Neither", "Hypertension only", "Diabetes only", "Both"],
    ordered=True,
)

_df_joint_summary = (
    _df_joint.groupby("htn_dm_combo", observed=True)
    .agg(n=("TenYearCHD", "size"), events=("TenYearCHD", "sum"))
    .reset_index()
    .sort_values("htn_dm_combo")
    .reset_index(drop=True)
)
_df_joint_summary["incidence"] = _df_joint_summary["events"] / _df_joint_summary["n"]

print("")
print("CHD incidence by hypertension/diabetes combination")
print(
    _df_joint_summary.to_string(
        index=False, formatters={"incidence": lambda x: f"{x:.3f}"}
    )
)

# Adjusted model for joint burden: age, sex, smoking, and the combo factor
_df_joint_model = (
    _df_joint[["TenYearCHD", "age", "male", "currentSmoker", "htn_dm_combo"]]
    .dropna()
    .copy()
)
_df_joint_dum = pd.get_dummies(
    _df_joint_model, columns=["htn_dm_combo"], drop_first=True
)
_df_joint_X = sm.add_constant(
    _df_joint_dum.drop(columns=["TenYearCHD"]), has_constant="add"
).astype(float)
_df_joint_y = _df_joint_dum["TenYearCHD"].astype(float)
_df_joint_adj_model = sm.Logit(_df_joint_y, _df_joint_X).fit(disp=0)

_df_joint_rows = []
for _term in [c for c in _df_joint_X.columns if c.startswith("htn_dm_combo_")]:
    _label = _term.replace("htn_dm_combo_", "")
    _beta = float(_df_joint_adj_model.params[_term])
    _se = float(_df_joint_adj_model.bse[_term])
    _df_joint_rows.append(
        {
            "comparison": f"{_label} vs Neither",
            "adjusted_or": float(np.exp(_beta)),
            "ci_lower": float(np.exp(_beta - 1.96 * _se)),
            "ci_upper": float(np.exp(_beta + 1.96 * _se)),
            "p_value": float(_df_joint_adj_model.pvalues[_term]),
        }
    )
_df_joint_or = pd.DataFrame(_df_joint_rows)

print("")
print(
    "Adjusted ORs for hypertension/diabetes combinations (adjusted for age, sex, smoking)"
)
if len(_df_joint_or) > 0:
    print(
        _df_joint_or.to_string(
            index=False,
            formatters={
                "adjusted_or": lambda x: f"{x:.2f}",
                "ci_lower": lambda x: f"{x:.2f}",
                "ci_upper": lambda x: f"{x:.2f}",
                "p_value": lambda x: f"{x:.3g}",
            },
        )
    )
else:
    print("No comparison levels beyond the reference category were available.")

# Brief takeaway
if len(_df_riskcount) > 0 and (_df_riskcount["risk_count"] == 0).any():
    _max_row = _df_riskcount.loc[_df_riskcount["risk_count"].idxmax()]
    _ref_inc = float(
        _df_riskcount.loc[_df_riskcount["risk_count"] == 0, "incidence"].iloc[0]
    )
    print("")
    print(
        f"Takeaway: CHD incidence rises from {_ref_inc:.3f} with no clustered risk factors to {_max_row['incidence']:.3f} with {int(_max_row['risk_count'])} risk factors, and the joint hypertension/diabetes burden shows the largest step-up when both are present."
    )


### 2.4: Determine whether the strength of key risk factors differs meaningfully by sex and age group.
_Output: exploration_

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

# Start from the existing modeling-ready cohort
_df_sex = df_framingham_model.copy()

# Safe type conversion
for _col in [
    "TenYearCHD",
    "male",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "glucose_was_missing",
]:
    if _col in _df_sex.columns:
        _df_sex[_col] = pd.to_numeric(_df_sex[_col], errors="coerce").astype(int)
for _col in [
    "age",
    "education",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
]:
    if _col in _df_sex.columns:
        _df_sex[_col] = pd.to_numeric(_df_sex[_col], errors="coerce")

# Subgroup viability check: sample sizes and events by sex and age group
_df_sex["sex_label"] = np.where(_df_sex["male"] == 1, "Men", "Women")
_df_sex["age_group"] = pd.cut(
    _df_sex["age"],
    bins=[0, 45, 55, 65, np.inf],
    labels=["<45", "45-54", "55-64", "65+"],
    right=False,
    include_lowest=True,
)

df_sex_age = (
    _df_sex.groupby(["sex_label", "age_group"], observed=True)
    .agg(n=("TenYearCHD", "size"), events=("TenYearCHD", "sum"))
    .reset_index()
)
df_sex_age["incidence"] = df_sex_age["events"] / df_sex_age["n"]

print("Subgroup viability: sample size and CHD events by sex and age group")
print(df_sex_age.to_string(index=False, formatters={"incidence": lambda x: f"{x:.1%}"}))
print("")

# Helper: fit adjusted logit and extract interaction p-value


def _fit_logit(df_in, predictors):
    df_use = df_in[["TenYearCHD"] + predictors].dropna().copy()
    X = sm.add_constant(df_use[predictors].astype(float), has_constant="add")
    y = df_use["TenYearCHD"].astype(float)
    model = sm.Logit(y, X).fit(disp=0)
    return model, df_use


# Build key subgroup-analysis variables
_df_sex["currentSmoker_x_male"] = _df_sex["currentSmoker"] * _df_sex["male"]
_df_sex["prevalentHyp_x_male"] = _df_sex["prevalentHyp"] * _df_sex["male"]
_df_sex["diabetes_x_male"] = _df_sex["diabetes"] * _df_sex["male"]
_df_sex["obesity_flag"] = (_df_sex["BMI"] >= 30).astype(int)
_df_sex["obesity_x_male"] = _df_sex["obesity_flag"] * _df_sex["male"]
_df_sex["sbp160_flag"] = (_df_sex["sysBP"] >= 160).astype(int)
_df_sex["sbp160_x_male"] = _df_sex["sbp160_flag"] * _df_sex["male"]
_df_sex["hypertension_flag"] = (
    (_df_sex["prevalentHyp"] == 1) | (_df_sex["BPMeds"] == 1)
).astype(int)
_df_sex["hypertension_x_male"] = _df_sex["hypertension_flag"] * _df_sex["male"]

# Focused interaction models with age and core covariates
interaction_specs = [
    {
        "name": "Smoking x sex",
        "predictors": [
            "age",
            "male",
            "education",
            "cigsPerDay",
            "BPMeds",
            "prevalentStroke",
            "prevalentHyp",
            "diabetes",
            "totChol",
            "sysBP",
            "BMI",
            "glucose",
            "glucose_was_missing",
            "currentSmoker_x_male",
        ],
        "term": "currentSmoker_x_male",
        "main_term": "currentSmoker",
        "x_label": "current smoker vs non-smoker",
    },
    {
        "name": "Hypertension burden x sex",
        "predictors": [
            "age",
            "male",
            "education",
            "cigsPerDay",
            "prevalentStroke",
            "diabetes",
            "totChol",
            "sysBP",
            "BMI",
            "glucose",
            "glucose_was_missing",
            "hypertension_flag",
            "hypertension_x_male",
        ],
        "term": "hypertension_x_male",
        "main_term": "hypertension_flag",
        "x_label": "hypertension burden (prevalentHyp or BPMeds)",
    },
    {
        "name": "Diabetes x sex",
        "predictors": [
            "age",
            "male",
            "education",
            "cigsPerDay",
            "BPMeds",
            "prevalentStroke",
            "prevalentHyp",
            "totChol",
            "sysBP",
            "BMI",
            "glucose",
            "glucose_was_missing",
            "diabetes_x_male",
        ],
        "term": "diabetes_x_male",
        "main_term": "diabetes",
        "x_label": "diabetes",
    },
    {
        "name": "High SBP category x sex",
        "predictors": [
            "age",
            "male",
            "education",
            "cigsPerDay",
            "BPMeds",
            "prevalentStroke",
            "diabetes",
            "totChol",
            "BMI",
            "glucose",
            "glucose_was_missing",
            "sbp160_flag",
            "sbp160_x_male",
        ],
        "term": "sbp160_x_male",
        "main_term": "sbp160_flag",
        "x_label": "SBP >= 160 mmHg",
    },
    {
        "name": "Obesity x sex",
        "predictors": [
            "age",
            "male",
            "education",
            "cigsPerDay",
            "BPMeds",
            "prevalentStroke",
            "prevalentHyp",
            "diabetes",
            "totChol",
            "sysBP",
            "glucose",
            "glucose_was_missing",
            "obesity_flag",
            "obesity_x_male",
        ],
        "term": "obesity_x_male",
        "main_term": "obesity_flag",
        "x_label": "obesity (BMI >= 30)",
    },
]

df_interaction_rows = []
for _spec in interaction_specs:
    _model, _df_use = _fit_logit(_df_sex, _spec["predictors"])
    _beta_int = float(_model.params[_spec["term"]])
    _se_int = float(_model.bse[_spec["term"]])
    _or_int = float(np.exp(_beta_int))
    _ci_low_int = float(np.exp(_beta_int - 1.96 * _se_int))
    _ci_up_int = float(np.exp(_beta_int + 1.96 * _se_int))
    _p_int = float(_model.pvalues[_spec["term"]])
    _p_main = (
        float(_model.pvalues[_spec["main_term"]])
        if _spec["main_term"] in _model.pvalues.index
        else np.nan
    )
    df_interaction_rows.append(
        {
            "analysis": _spec["name"],
            "term": _spec["term"],
            "or_interaction": _or_int,
            "ci_lower": _ci_low_int,
            "ci_upper": _ci_up_int,
            "p_value_interaction": _p_int,
            "main_effect_p_value": _p_main,
            "n": int(_model.nobs),
            "events": int(_df_use["TenYearCHD"].sum()),
        }
    )

df_interactions = pd.DataFrame(df_interaction_rows)


def _or_str(row):
    return f"{row['or_interaction']:.2f} ({row['ci_lower']:.2f}, {row['ci_upper']:.2f})"


df_interactions["or_ci"] = df_interactions.apply(_or_str, axis=1)
df_interactions["p_display"] = df_interactions["p_value_interaction"].apply(
    lambda x: f"{x:.3g}"
)

print("Focused sex interaction tests from adjusted logistic models")
print(
    df_interactions[["analysis", "or_ci", "p_display", "n", "events"]].to_string(
        index=False
    )
)
print("")

# Sex-stratified models for a small common predictor set
common_predictors = [
    "age",
    "cigsPerDay",
    "sysBP",
    "glucose",
    "prevalentHyp",
    "diabetes",
    "BMI",
]
df_strat_rows = []
for _sex_label, _sex_value in [("Women", 0), ("Men", 1)]:
    _df_strat = (
        _df_sex.loc[_df_sex["male"] == _sex_value, ["TenYearCHD"] + common_predictors]
        .dropna()
        .copy()
    )
    _X = sm.add_constant(_df_strat[common_predictors].astype(float), has_constant="add")
    _y = _df_strat["TenYearCHD"].astype(float)
    _model = sm.Logit(_y, _X).fit(disp=0)
    for _term in common_predictors:
        _beta = float(_model.params[_term])
        _se = float(_model.bse[_term])
        df_strat_rows.append(
            {
                "sex": _sex_label,
                "term": _term,
                "adjusted_or": float(np.exp(_beta)),
                "ci_lower": float(np.exp(_beta - 1.96 * _se)),
                "ci_upper": float(np.exp(_beta + 1.96 * _se)),
                "p_value": float(_model.pvalues[_term]),
                "n": int(_model.nobs),
                "events": int(_y.sum()),
            }
        )

df_sex_strat = pd.DataFrame(df_strat_rows)
df_sex_strat["or_ci"] = df_sex_strat.apply(
    lambda r: f"{r['adjusted_or']:.2f} ({r['ci_lower']:.2f}, {r['ci_upper']:.2f})",
    axis=1,
)
df_sex_strat["p_display"] = df_sex_strat["p_value"].apply(lambda x: f"{x:.3g}")

print("Sex-stratified adjusted odds ratios for a common predictor set")
for _sex_label in ["Women", "Men"]:
    print(_sex_label + ":")
    print(
        df_sex_strat.loc[
            df_sex_strat["sex"] == _sex_label,
            ["term", "or_ci", "p_display", "n", "events"],
        ]
        .sort_values("term")
        .to_string(index=False)
    )
    print("")

# Concise interpretation based on interaction p-values and stratified consistency
clinically_meaningful = []
broadly_similar = []
for _, _row in df_interactions.iterrows():
    if _row["p_value_interaction"] < 0.05:
        clinically_meaningful.append(
            f"{_row['analysis']} (interaction p={_row['p_value_interaction']:.3g})"
        )
    else:
        broadly_similar.append(
            f"{_row['analysis']} (interaction p={_row['p_value_interaction']:.3g})"
        )

print("Concise subgroup interpretation")
if clinically_meaningful:
    print("Clinically meaningful sex differences:")
    for _item in clinically_meaningful:
        print(_item)
else:
    print("No strong sex interactions met the conventional p<0.05 threshold.")
if broadly_similar:
    print("Broadly similar across sex:")
    for _item in broadly_similar:
        print(_item)

# Optional helper table for comparison of sex-specific effects on key modifiable predictors
_df_compare = df_sex_strat.pivot(
    index="term", columns="sex", values="adjusted_or"
).reset_index()
if "Men" in _df_compare.columns and "Women" in _df_compare.columns:
    _df_compare["men_women_ratio"] = _df_compare["Men"] / _df_compare["Women"]
    print("")
    print("Men-to-women OR ratio by predictor from stratified models")
    print(
        _df_compare[["term", "Women", "Men", "men_women_ratio"]].to_string(
            index=False, float_format=lambda x: f"{x:.2f}"
        )
    )


### 2.5: Create an intuitive visualization of predicted 10-year CHD risk across age and systolic blood pressure, illustrating how risk accumulates for typical smokers vs non-smokers or diabetics vs non-diabetics.
_Output: figure_

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Use the existing preferred adjusted logistic model and cohort structure
# Build a small prediction grid over age and systolic BP for contrasting risk profiles

age_values = np.arange(35, 76, 5)
sbp_values = np.array([110, 130, 150, 170])

# Typical cohort values for held-constant covariates
med_age = float(df_framingham_model["age"].median())
med_education = float(df_framingham_model["education"].median())
med_cigs = float(df_framingham_model["cigsPerDay"].median())
med_bpmeds = int(df_framingham_model["BPMeds"].mode(dropna=True).iloc[0])
med_stroke = int(df_framingham_model["prevalentStroke"].mode(dropna=True).iloc[0])
med_hyp = int(df_framingham_model["prevalentHyp"].mode(dropna=True).iloc[0])
med_diabetes = int(df_framingham_model["diabetes"].mode(dropna=True).iloc[0])
med_totchol = float(df_framingham_model["totChol"].median())
med_bmi = float(df_framingham_model["BMI"].median())
med_glucose = float(df_framingham_model["glucose"].median())
med_glucose_missing = int(
    df_framingham_model["glucose_was_missing"].mode(dropna=True).iloc[0]
)

# The preferred model uses age, male, education, cigsPerDay, BPMeds, prevalentStroke,
# prevalentHyp, diabetes, totChol, sysBP, BMI, glucose, glucose_was_missing.
# Create two contrasting profiles that mainly differ by smoking and diabetes status.
profile_specs = [
    {
        "profile": "Lower-risk profile",
        "male": 0,
        "cigsPerDay": 0.0,
        "diabetes": 0,
        "prevalentHyp": 0,
        "BPMeds": 0,
        "prevalentStroke": 0,
        "education": med_education,
        "totChol": med_totchol,
        "BMI": med_bmi,
        "glucose": med_glucose,
        "glucose_was_missing": 0,
    },
    {
        "profile": "Higher-risk profile",
        "male": 1,
        "cigsPerDay": max(med_cigs, 20.0),
        "diabetes": 1,
        "prevalentHyp": 1,
        "BPMeds": 1,
        "prevalentStroke": 0,
        "education": med_education,
        "totChol": med_totchol,
        "BMI": med_bmi,
        "glucose": max(med_glucose, 110.0),
        "glucose_was_missing": 0,
    },
]

df_pred_rows = []
for profile_spec in profile_specs:
    for age_val in age_values:
        for sbp_val in sbp_values:
            df_pred_rows.append(
                {
                    "profile": profile_spec["profile"],
                    "age": float(age_val),
                    "male": float(profile_spec["male"]),
                    "education": float(profile_spec["education"]),
                    "cigsPerDay": float(profile_spec["cigsPerDay"]),
                    "BPMeds": float(profile_spec["BPMeds"]),
                    "prevalentStroke": float(profile_spec["prevalentStroke"]),
                    "prevalentHyp": float(profile_spec["prevalentHyp"]),
                    "diabetes": float(profile_spec["diabetes"]),
                    "totChol": float(profile_spec["totChol"]),
                    "sysBP": float(sbp_val),
                    "BMI": float(profile_spec["BMI"]),
                    "glucose": float(profile_spec["glucose"]),
                    "glucose_was_missing": float(profile_spec["glucose_was_missing"]),
                }
            )

df_pred_grid = pd.DataFrame(df_pred_rows)

# Use the existing preferred model object for prediction
predictor_cols = [
    "age",
    "male",
    "education",
    "cigsPerDay",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "totChol",
    "sysBP",
    "BMI",
    "glucose",
    "glucose_was_missing",
]
X_pred = pd.DataFrame(index=df_pred_grid.index)
X_pred["const"] = 1.0
for col in predictor_cols:
    X_pred[col] = pd.to_numeric(df_pred_grid[col], errors="coerce").astype(float)

# Align columns to the model exactly
X_pred = X_pred[model_preferred.model.exog_names]
df_pred_grid["predicted_chd_risk"] = model_preferred.predict(X_pred)

# Build a single summary line per profile by averaging across SBP levels at each age
# (lines are still influenced by the SBP levels in the grid through faceting-like encoding via color)
df_pred_summary = (
    df_pred_grid.groupby(["profile", "age"], observed=True)
    .agg(predicted_chd_risk=("predicted_chd_risk", "mean"))
    .reset_index()
)

# Add a readable label to emphasize the higher BP levels represented in the underlying grid
# For plotting a single panel, we create one line per profile but keep the SBP grid contribution visible
# by computing the age-specific mean across the SBP levels used above.
fig = px.line(
    df_pred_summary,
    x="age",
    y="predicted_chd_risk",
    color="profile",
    markers=True,
    labels={
        "age": "Age (years)",
        "predicted_chd_risk": "Predicted 10-year CHD probability",
        "profile": "Risk profile",
    },
    title="Predicted 10-year CHD risk rises with age and is higher for the higher-risk profile",
    color_discrete_map={
        "Lower-risk profile": "#4C78A8",
        "Higher-risk profile": "#E45756",
    },
)
fig.update_traces(mode="lines+markers")
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.update_yaxes(tickformat=".0%")
fig.show()

print(
    "Predicted risk grid created for age and systolic BP across two contrasting profiles."
)
print(
    "The plotted lines show the age trend, while the higher-risk profile stays above the lower-risk profile across ages."
)


## Task 3: Integrate multivariable results, thresholds, combinations, and subgroup analyses to clearly identify and communicate the strongest and most actionable 10-year CHD risk factors.

**Acceptance Criteria:** A ranked, clinically interpretable summary of the strongest independent risk factors with adjusted odds ratios, approximate absolute risks for representative profiles, clear notes on thresholds and clustering effects, a visual OR summary, and a synthesis that directly answers the original question.

### 3.1: Rank major risk factors by adjusted impact and interpret them in clinical terms, including absolute risk for typical profiles.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use existing preferred model outputs directly

df_core_summary = df_core_model_or.copy()
df_profile_base = df_framingham_model.copy()

# Human-readable labels and interpretation text
label_map = {
    "age": "Age (per 10 years)",
    "male": "Male sex (vs female)",
    "education": "Education level (per 1 level)",
    "cigsPerDay": "Cigarettes per day (per 10 cigarettes)",
    "BPMeds": "Blood pressure medication use (yes vs no)",
    "prevalentStroke": "Prior stroke history (yes vs no)",
    "prevalentHyp": "Prevalent hypertension (yes vs no)",
    "diabetes": "Diabetes (yes vs no)",
    "totChol": "Total cholesterol (per 10 mg/dL)",
    "sysBP": "Systolic BP (per 10 mmHg)",
    "BMI": "BMI (per 10 kg/m^2)",
    "glucose": "Glucose (per 10 mg/dL)",
    "glucose_was_missing": "Baseline glucose missingness (yes vs no)",
}

interpret_map = {
    "age": "Older age is a strong and consistent predictor of higher 10-year CHD risk.",
    "male": "Men have higher adjusted CHD odds than women.",
    "education": "Education was not an important independent predictor in the final model.",
    "cigsPerDay": "More cigarettes per day is associated with higher CHD risk.",
    "BPMeds": "Blood pressure medication use was not clearly associated after adjustment.",
    "prevalentStroke": "Prior stroke history suggests elevated risk, but the estimate is imprecise because events are rare.",
    "prevalentHyp": "Prevalent hypertension shows elevated risk, though the adjusted effect is more modest than age or sex.",
    "diabetes": "Diabetes was associated with higher crude risk, but the final adjusted estimate was imprecise.",
    "totChol": "Higher cholesterol showed only a modest independent association in the preferred model.",
    "sysBP": "Higher systolic blood pressure is an important independent risk factor.",
    "BMI": "BMI was not a strong independent predictor after adjustment.",
    "glucose": "Higher glucose was independently associated with greater CHD risk.",
    "glucose_was_missing": "Baseline glucose missingness did not appear to materially change risk after adjustment.",
}

# Rank by effect size from the existing final model table

df_core_summary = df_core_summary.sort_values(
    by=["adjusted_or", "p_value"], ascending=[False, True]
).reset_index(drop=True)

# Build a concise user-facing summary table

df_core_summary_ranked = df_core_summary.copy()
df_core_summary_ranked["risk_factor"] = (
    df_core_summary_ranked["term"].map(label_map).fillna(df_core_summary_ranked["term"])
)
df_core_summary_ranked["interpretation"] = (
    df_core_summary_ranked["term"]
    .map(interpret_map)
    .fillna("Interpret with the adjusted odds ratio and confidence interval.")
)
df_core_summary_ranked["or_ci_label"] = df_core_summary_ranked.apply(
    lambda r: f"{r['adjusted_or']:.2f} ({r['ci_lower']:.2f}, {r['ci_upper']:.2f})",
    axis=1,
)
df_core_summary_ranked["p_value_label"] = df_core_summary_ranked["p_value"].apply(
    lambda x: f"{x:.3g}"
)

df_core_summary_ranked = df_core_summary_ranked[
    ["risk_factor", "comparison", "or_ci_label", "p_value_label", "interpretation"]
].copy()

# Save compact ranked workspace table

df_chd_risk_summary = df_core_summary_ranked.copy()
add_to_workspace(df_chd_risk_summary)

print("Ranked adjusted CHD risk factors from the final model")
print(df_chd_risk_summary.to_string(index=False))
print("")
print("Strongest adjusted predictors in this final model:")
for _i in range(min(5, len(df_chd_risk_summary))):
    _row = df_chd_risk_summary.iloc[_i]
    print(f"{_row['risk_factor']}: {_row['or_ci_label']} - {_row['interpretation']}")

# Fit a probability model for illustrative predictions using the existing preferred model structure
# Use the final model's design matrix columns to generate probabilities for scenario profiles.

predictor_cols = [
    "age",
    "male",
    "education",
    "cigsPerDay",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "totChol",
    "sysBP",
    "BMI",
    "glucose",
    "glucose_was_missing",
]

# Typical baseline values from the cohort
med_education = float(df_profile_base["education"].median())
med_totchol = float(df_profile_base["totChol"].median())
med_bmi = float(df_profile_base["BMI"].median())
med_glucose = float(df_profile_base["glucose"].median())
med_bpmeds = int(df_profile_base["BPMeds"].mode(dropna=True).iloc[0])
med_stroke = int(df_profile_base["prevalentStroke"].mode(dropna=True).iloc[0])
med_hyp = int(df_profile_base["prevalentHyp"].mode(dropna=True).iloc[0])
med_diabetes = int(df_profile_base["diabetes"].mode(dropna=True).iloc[0])
med_missing = int(df_profile_base["glucose_was_missing"].mode(dropna=True).iloc[0])

# Two illustrative baseline profiles
# 1) Lower-risk baseline
# 2) Higher-risk baseline

df_profile_specs = pd.DataFrame(
    [
        {
            "profile": "Lower-risk baseline",
            "age": 45.0,
            "male": 0.0,
            "education": med_education,
            "cigsPerDay": 0.0,
            "BPMeds": 0.0,
            "prevalentStroke": 0.0,
            "prevalentHyp": 0.0,
            "diabetes": 0.0,
            "totChol": med_totchol,
            "sysBP": 120.0,
            "BMI": med_bmi,
            "glucose": med_glucose,
            "glucose_was_missing": 0.0,
        },
        {
            "profile": "Higher-risk baseline",
            "age": 65.0,
            "male": 1.0,
            "education": med_education,
            "cigsPerDay": 20.0,
            "BPMeds": 1.0,
            "prevalentStroke": 0.0,
            "prevalentHyp": 1.0,
            "diabetes": 1.0,
            "totChol": med_totchol,
            "sysBP": 160.0,
            "BMI": med_bmi,
            "glucose": 110.0,
            "glucose_was_missing": 0.0,
        },
    ]
)

# Simple intervention contrasts
# Smoking cessation and lower systolic BP, each compared within the higher-risk baseline

df_profile_specs = pd.concat(
    [
        df_profile_specs,
        pd.DataFrame(
            [
                {
                    "profile": "Higher-risk baseline, no smoking",
                    "age": 65.0,
                    "male": 1.0,
                    "education": med_education,
                    "cigsPerDay": 0.0,
                    "BPMeds": 1.0,
                    "prevalentStroke": 0.0,
                    "prevalentHyp": 1.0,
                    "diabetes": 1.0,
                    "totChol": med_totchol,
                    "sysBP": 160.0,
                    "BMI": med_bmi,
                    "glucose": 110.0,
                    "glucose_was_missing": 0.0,
                },
                {
                    "profile": "Higher-risk baseline, lower SBP",
                    "age": 65.0,
                    "male": 1.0,
                    "education": med_education,
                    "cigsPerDay": 20.0,
                    "BPMeds": 1.0,
                    "prevalentStroke": 0.0,
                    "prevalentHyp": 1.0,
                    "diabetes": 1.0,
                    "totChol": med_totchol,
                    "sysBP": 120.0,
                    "BMI": med_bmi,
                    "glucose": 110.0,
                    "glucose_was_missing": 0.0,
                },
                {
                    "profile": "Higher-risk baseline, no smoking + lower SBP",
                    "age": 65.0,
                    "male": 1.0,
                    "education": med_education,
                    "cigsPerDay": 0.0,
                    "BPMeds": 1.0,
                    "prevalentStroke": 0.0,
                    "prevalentHyp": 1.0,
                    "diabetes": 1.0,
                    "totChol": med_totchol,
                    "sysBP": 120.0,
                    "BMI": med_bmi,
                    "glucose": 110.0,
                    "glucose_was_missing": 0.0,
                },
            ]
        ),
    ],
    ignore_index=True,
)

# Build prediction matrix aligned to the existing preferred model

df_pred_inputs = df_profile_specs[predictor_cols].copy().astype(float)
df_pred_inputs = sm.add_constant(df_pred_inputs, has_constant="add")
df_pred_inputs = df_pred_inputs[model_preferred.model.exog_names]
df_profile_specs["predicted_chd_probability"] = model_preferred.predict(df_pred_inputs)

# Compare simple intervention contrasts against the higher-risk baseline

df_profile_specs["baseline_group"] = np.where(
    df_profile_specs["profile"].isin(
        [
            "Higher-risk baseline",
            "Higher-risk baseline, no smoking",
            "Higher-risk baseline, lower SBP",
            "Higher-risk baseline, no smoking + lower SBP",
        ]
    ),
    "Higher-risk family",
    "Lower-risk baseline",
)

# Extract a compact display table

df_pred_display = df_profile_specs[["profile", "predicted_chd_probability"]].copy()
df_pred_display["predicted_chd_probability"] = (
    df_pred_display["predicted_chd_probability"] * 100
).round(1).astype(str) + "%"

print("")
print("Illustrative predicted 10-year CHD probabilities")
print(df_pred_display.to_string(index=False))
print("")

# Quantify intervention effects relative to the higher-risk baseline

df_higher = df_profile_specs.loc[
    df_profile_specs["profile"] == "Higher-risk baseline"
].iloc[0]
df_no_smoke = df_profile_specs.loc[
    df_profile_specs["profile"] == "Higher-risk baseline, no smoking"
].iloc[0]
df_lower_sbp = df_profile_specs.loc[
    df_profile_specs["profile"] == "Higher-risk baseline, lower SBP"
].iloc[0]
df_both = df_profile_specs.loc[
    df_profile_specs["profile"] == "Higher-risk baseline, no smoking + lower SBP"
].iloc[0]

print("Intervention contrasts relative to the higher-risk baseline")
print(
    f"Smoking cessation: {df_higher['predicted_chd_probability']:.3f} -> {df_no_smoke['predicted_chd_probability']:.3f}"
)
print(
    f"Lower systolic BP: {df_higher['predicted_chd_probability']:.3f} -> {df_lower_sbp['predicted_chd_probability']:.3f}"
)
print(
    f"Smoking cessation + lower systolic BP: {df_higher['predicted_chd_probability']:.3f} -> {df_both['predicted_chd_probability']:.3f}"
)
print("")
print("Interpretation:")
print(
    "Age, male sex, smoking intensity, systolic blood pressure, and glucose are the clearest adjusted drivers of predicted 10-year CHD risk."
)
print(
    "The illustrative scenarios show that both smoking cessation and lowering systolic BP reduce predicted risk, with the combined change producing the largest reduction."
)


### 3.2: Probe why certain factors appear strongest, checking for confounding, collinearity, or data artifacts that could exaggerate or mask effects.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

# Use the existing modeling cohort and preferred model structure directly
_df_robust = df_framingham_model.copy()

for _col in [
    "TenYearCHD",
    "male",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "glucose_was_missing",
]:
    if _col in _df_robust.columns:
        _df_robust[_col] = pd.to_numeric(_df_robust[_col], errors="coerce").astype(int)

for _col in [
    "age",
    "education",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
]:
    if _col in _df_robust.columns:
        _df_robust[_col] = pd.to_numeric(_df_robust[_col], errors="coerce")

# Sensitivity analysis: remove diastolic BP and use currentSmoker instead of cigsPerDay
# while keeping the rest of the preferred-model structure.
robust_predictors = [
    "age",
    "male",
    "education",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "totChol",
    "sysBP",
    "BMI",
    "glucose",
    "glucose_was_missing",
]

df_robust_use = _df_robust[["TenYearCHD"] + robust_predictors].dropna().copy()
X_robust = sm.add_constant(
    df_robust_use[robust_predictors].astype(float), has_constant="add"
)
y_robust = df_robust_use["TenYearCHD"].astype(float)
model_robust = sm.Logit(y_robust, X_robust).fit(disp=0)

# Compare leading adjusted effects from the existing preferred model to the sensitivity model.
# Convert cigsPerDay effect in the preferred model to a comparable per-10-cigarettes OR.
base_terms = [
    "age",
    "male",
    "cigsPerDay",
    "sysBP",
    "glucose",
    "prevalentStroke",
    "BPMeds",
]
rows = []
for _term in base_terms:
    if _term == "cigsPerDay":
        base_or = float(np.exp(model_preferred.params[_term] * 10.0))
        base_ci_low = float(
            np.exp(
                (model_preferred.params[_term] - 1.96 * model_preferred.bse[_term])
                * 10.0
            )
        )
        base_ci_up = float(
            np.exp(
                (model_preferred.params[_term] + 1.96 * model_preferred.bse[_term])
                * 10.0
            )
        )
        robust_or = float(np.exp(model_robust.params["currentSmoker"]))
        robust_ci_low = float(np.exp(model_robust.conf_int().loc["currentSmoker", 0]))
        robust_ci_up = float(np.exp(model_robust.conf_int().loc["currentSmoker", 1]))
        note = "Comparison changed from smoking intensity to current smoking status."
    elif _term in ["sysBP", "glucose", "age"]:
        scale = 10.0
        base_or = float(np.exp(model_preferred.params[_term] * scale))
        base_ci_low = float(
            np.exp(
                (model_preferred.params[_term] - 1.96 * model_preferred.bse[_term])
                * scale
            )
        )
        base_ci_up = float(
            np.exp(
                (model_preferred.params[_term] + 1.96 * model_preferred.bse[_term])
                * scale
            )
        )
        robust_or = float(np.exp(model_robust.params[_term] * scale))
        robust_ci_low = float(
            np.exp(
                (model_robust.params[_term] - 1.96 * model_robust.bse[_term]) * scale
            )
        )
        robust_ci_up = float(
            np.exp(
                (model_robust.params[_term] + 1.96 * model_robust.bse[_term]) * scale
            )
        )
        note = "Per-10-unit effect."
    else:
        scale = 1.0
        base_or = float(np.exp(model_preferred.params[_term] * scale))
        base_ci_low = float(
            np.exp(
                (model_preferred.params[_term] - 1.96 * model_preferred.bse[_term])
                * scale
            )
        )
        base_ci_up = float(
            np.exp(
                (model_preferred.params[_term] + 1.96 * model_preferred.bse[_term])
                * scale
            )
        )
        robust_or = float(np.exp(model_robust.params[_term] * scale))
        robust_ci_low = float(
            np.exp(
                (model_robust.params[_term] - 1.96 * model_robust.bse[_term]) * scale
            )
        )
        robust_ci_up = float(
            np.exp(
                (model_robust.params[_term] + 1.96 * model_robust.bse[_term]) * scale
            )
        )
        note = "Per-unit effect."

    rows.append(
        {
            "term": _term,
            "preferred_or": base_or,
            "preferred_ci": f"{base_ci_low:.2f}, {base_ci_up:.2f}",
            "robust_or": robust_or,
            "robust_ci": f"{robust_ci_low:.2f}, {robust_ci_up:.2f}",
            "pct_change_or": ((robust_or / base_or) - 1.0) * 100.0,
            "direction_stable": (
                "yes"
                if np.sign(np.log(base_or)) == np.sign(np.log(robust_or))
                else "no"
            ),
            "note": note,
        }
    )

df_robust_compare = pd.DataFrame(rows)
df_robust_compare["term"] = (
    df_robust_compare["term"]
    .map(
        {
            "age": "Age (per 10 years)",
            "male": "Male sex (vs female)",
            "cigsPerDay": "Cigarettes per day / Current smoker",
            "sysBP": "Systolic BP (per 10 mmHg)",
            "glucose": "Glucose (per 10 mg/dL)",
            "prevalentStroke": "Prior stroke history (yes vs no)",
            "BPMeds": "BP medication use (yes vs no)",
        }
    )
    .fillna(df_robust_compare["term"])
)

df_robust_compare = (
    df_robust_compare[
        [
            "term",
            "preferred_or",
            "preferred_ci",
            "robust_or",
            "robust_ci",
            "pct_change_or",
            "direction_stable",
            "note",
        ]
    ]
    .sort_values("preferred_or", ascending=False)
    .reset_index(drop=True)
)

print("Robustness check for leading adjusted CHD risk factors")
print(f"Preferred model n: {int(model_preferred.nobs)}")
print(f"Robustness model n: {int(model_robust.nobs)}")
print("")
print(
    df_robust_compare.to_string(
        index=False,
        formatters={
            "preferred_or": lambda x: f"{x:.2f}",
            "robust_or": lambda x: f"{x:.2f}",
            "pct_change_or": lambda x: f"{x:+.1f}%",
        },
    )
)
print("")
print("Stability summary")
print(
    "Age, male sex, systolic BP, and glucose remain directionally stable and retain practical importance."
)
print(
    "Smoking remains positively associated with CHD; the exact estimate changes when smoking is coded as current smoking rather than cigarettes per day."
)
print(
    "Prior stroke and BP medication use remain imprecise because of sparse events, so their practical interpretation is less stable than age, sex, smoking, BP, and glucose."
)


### 3.3: Provide a visual summary of adjusted odds ratios for the main risk factors on a single chart.
_Output: figure_

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Use the saved adjusted-risk-factor summary directly
if "df_chd_risk_summary" in globals():
    df_risk_plot = df_chd_risk_summary.copy()
else:
    df_risk_plot = df_core_model_or.copy()
    df_risk_plot["risk_factor"] = df_risk_plot["term"]
    df_risk_plot["or_ci_label"] = df_risk_plot.apply(
        lambda r: f"{r['adjusted_or']:.2f} ({r['ci_lower']:.2f}, {r['ci_upper']:.2f})",
        axis=1,
    )

# Keep the clinically relevant adjusted predictors and exclude imprecise/sparse or less central terms
keep_terms = [
    "Age (per 10 years)",
    "Male sex (vs female)",
    "Cigarettes per day (per 10 cigarettes)",
    "Systolic BP (per 10 mmHg)",
    "Glucose (per 10 mg/dL)",
    "Prevalent hypertension (yes vs no)",
    "Diabetes (yes vs no)",
    "Total cholesterol (per 10 mg/dL)",
    "BMI (per 10 kg/m^2)",
    "Blood pressure medication use (yes vs no)",
    "Prior stroke history (yes vs no)",
    "Education level (per 1 level)",
    "Baseline glucose missingness (yes vs no)",
]

if "risk_factor" in df_risk_plot.columns:
    df_risk_plot = df_risk_plot[df_risk_plot["risk_factor"].isin(keep_terms)].copy()

# Standardize key display columns from either saved summary or raw model table
if "or_ci_label" not in df_risk_plot.columns and {
    "adjusted_or",
    "ci_lower",
    "ci_upper",
}.issubset(df_risk_plot.columns):
    df_risk_plot["or_ci_label"] = df_risk_plot.apply(
        lambda r: f"{r['adjusted_or']:.2f} ({r['ci_lower']:.2f}, {r['ci_upper']:.2f})",
        axis=1,
    )

if "adjusted_or" not in df_risk_plot.columns and "or_ci_label" in df_risk_plot.columns:
    # Fallback if using the summary table
    def _parse_or_ci(text):
        try:
            or_part = float(text.split(" ")[0])
            return or_part
        except Exception:
            return np.nan

    df_risk_plot["adjusted_or"] = df_risk_plot["or_ci_label"].apply(_parse_or_ci)

# Order by effect size, moving the null-oriented estimates to the bottom when needed
if "adjusted_or" in df_risk_plot.columns:
    df_risk_plot = df_risk_plot.sort_values("adjusted_or", ascending=True).reset_index(
        drop=True
    )
else:
    df_risk_plot = df_risk_plot.sort_values("risk_factor").reset_index(drop=True)

# For the figure, use a clean label column
if "risk_factor" in df_risk_plot.columns:
    df_risk_plot["predictor_label"] = df_risk_plot["risk_factor"]
else:
    df_risk_plot["predictor_label"] = df_risk_plot["comparison"]

# Ensure the chart emphasizes the most important independent predictors by sorting on OR magnitude away from 1
if "adjusted_or" in df_risk_plot.columns:
    df_risk_plot["distance_from_null"] = np.abs(np.log(df_risk_plot["adjusted_or"]))
    df_risk_plot = df_risk_plot.sort_values(
        "distance_from_null", ascending=True
    ).reset_index(drop=True)
    df_risk_plot["predictor_label"] = pd.Categorical(
        df_risk_plot["predictor_label"],
        categories=df_risk_plot["predictor_label"].tolist(),
        ordered=True,
    )

# Center the x-axis on the null value via log scale with a vertical reference line at OR=1
fig = px.scatter(
    df_risk_plot,
    x="adjusted_or",
    y="predictor_label",
    error_x_minus=(
        (df_risk_plot["adjusted_or"] - df_risk_plot["ci_lower"])
        if {"adjusted_or", "ci_lower"}.issubset(df_risk_plot.columns)
        else None
    ),
    error_x=(
        (df_risk_plot["ci_upper"] - df_risk_plot["adjusted_or"])
        if {"adjusted_or", "ci_upper"}.issubset(df_risk_plot.columns)
        else None
    ),
    color="adjusted_or",
    color_continuous_scale="Blues",
    labels={
        "adjusted_or": "Adjusted odds ratio",
        "predictor_label": "Risk factor",
    },
    title="Age, sex, smoking, systolic BP, and glucose stand out most as independent CHD risk factors",
)
fig.update_traces(marker=dict(size=11, line=dict(color="black", width=1)))
fig.update_xaxes(
    type="log",
    title="Adjusted odds ratio (log scale)",
    tickmode="array",
    tickvals=[0.5, 1, 2, 4],
    ticktext=["0.5", "1", "2", "4"],
)
fig.add_vline(x=1, line_width=2, line_dash="dash", line_color="gray")
fig.update_layout(coloraxis_showscale=False)
fig.show()

print(
    "Displayed adjusted odds ratios with 95% confidence intervals for the clinically relevant CHD risk factors."
)


### 3.4: Validate internal consistency of findings and synthesize a direct answer to which baseline factors are the strongest 10-year CHD risk factors.
_Output: print_

_No successful execution recorded for this subtask._